#  Topic Modeling and Trend Detection in Large Text Corpora

Topic modeling is a technique in natural language processing (NLP) and machine learning that aims to uncover latent thematic structures within a collection of texts. Topic modelling is a system learning technique that robotically discovers the principle themes or "topics" representing a huge document collection. The intention of topic modelling is to discover the hidden semantic systems within textual content facts, permitting customers to arrange, apprehend, and summarize the data in a manner that is each green and insightful.




## Arxiv Dataset
For this project we will use the Arxiv dataset which is a mirror of the original ArXiv data. For nearly 30 years, ArXiv has served the public and research communities by providing open access to scholarly articles, from the vast branches of physics to the many subdisciplines of computer science to everything in between, including math, statistics, electrical engineering, quantitative biology, and economics. This rich corpus of information offers significant, but sometimes overwhelming depth. Since the full arXiv dataset is quite large (approximately 1.1 TB and continuously growing), for the purposes of this project I will be using only the first 40,000 rows to reduce resource usage and ensure faster processing during development and experimentation.

### Install Bertopic

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

### Import the necessary libraries

In [ ]:
from umap import UMAP
import pandas as pd
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired,MaximalMarginalRelevance
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.metrics import silhouette_score,calinski_harabasz_score,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

### Fetch dataset

In [ ]:
dataset = pd.read_csv('/content/drive/MyDrive/arxiv-40k.csv')

In [ ]:
dataset

,update_date,title,journal-ref,submitter,authors,doi,comments,categories,abstract,id,license,report-no,versions,authors_parsed
0,2008-11-26,Calculation of prompt diphoton production cros...,"Phys.Rev.D76:013009,2007",Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",10.1103/PhysRevD.76.013009,"37 pages, 15 figures; published version",hep-ph,A fully differential calculation in perturba...,704.0001,NaN,ANL-HEP-PR-07-12,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Balázs', 'C.', ''], ['Berger', 'E. L.', '']..."
1,2008-12-13,Sparsity-certifying Graph Decompositions,NaN,Louis Theran,Ileana Streinu and Louis Theran,NaN,To appear in Graphs and Combinatorics,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-...",704.0002,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Streinu', 'Ileana', ''], ['Theran', 'Louis'..."
2,2008-01-13,The evolution of the Earth-Moon system based o...,NaN,Hongjun Pan,Hongjun Pan,NaN,"23 pages, 3 figures",physics.gen-ph,The evolution of Earth-Moon system is descri...,704.0003,NaN,NaN,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...","[['Pan', 'Hongjun', '']]"
3,2007-05-23,A determinant of Stirling cycle numbers counts...,NaN,David Callan,David Callan,NaN,11 pages,math.CO,We show that a determinant of Stirling cycle...,704.0004,NaN,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Callan', 'David', '']]"
4,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,"Illinois J. Math. 52 (2008) no.2, 681-689",Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,NaN,NaN,math.CA math.FA,In this paper we show how to compute the $\L...,704.0005,NaN,NaN,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Abu-Shammala', 'Wael', ''], ['Torchinsky', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,2008-11-26,Flavorful Supersymmetry,"Phys.Rev.D77:075006,2008",Yasunori Nomura,"Yasunori Nomura, Michele Papucci, Daniel Stola...",10.1103/PhysRevD.77.075006,"20 pages; typos corrected, comments added, to ...",hep-ph,Weak scale supersymmetry provides elegant so...,712.2074,NaN,UCB-PTH-07/25,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Nomura', 'Yasunori', ''], ['Papucci', 'Mich..."
39996,2011-07-20,"($\ell,0)$-Carter partitions, a generating fun...","Electronic Journal of Combinatorics, Volume 15...",Chris Berg,"Chris Berg, Monica Vazirani",NaN,NaN,math.CO math.RT,In this paper we give an alternate combinato...,712.2075,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Berg', 'Chris', ''], ['Vazirani', 'Monica',..."
39997,2007-12-20,On the irreducible representations of a finite...,NaN,Benjamin Steinberg,"Olexandr Ganyushkin, Volodymyr Mazorchuk and B...",NaN,NaN,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ...",712.2076,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Ganyushkin', 'Olexandr', ''], ['Mazorchuk',..."
39998,2009-11-13,The Formation of Constellation III in the Larg...,NaN,Jason Harris,Jason Harris and Dennis Zaritsky,10.1071/AS07037,Accepted for publication in the Publications o...,astro-ph,We present a detailed reconstruction of the ...,712.2077,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Harris', 'Jason', ''], ['Zaritsky', 'Dennis..."


### Data Preprocessing

In [ ]:
dataset.isnull().sum()

,0
update_date,0
title,0
journal-ref,19270
submitter,0
authors,0
doi,15629
comments,4709
categories,0
abstract,0
id,0


In [ ]:
dataset = dataset.drop(columns=['update_date','title','journal-ref','submitter','authors','doi','comments','id','license','report-no','versions','authors_parsed'],axis=1)
dataset

,categories,abstract
0,hep-ph,A fully differential calculation in perturba...
1,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-..."
2,physics.gen-ph,The evolution of Earth-Moon system is descri...
3,math.CO,We show that a determinant of Stirling cycle...
4,math.CA math.FA,In this paper we show how to compute the $\L...
...,...,...
39995,hep-ph,Weak scale supersymmetry provides elegant so...
39996,math.CO math.RT,In this paper we give an alternate combinato...
39997,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ..."
39998,astro-ph,We present a detailed reconstruction of the ...


In [ ]:
threshold = 100
val_counts = dataset['categories'].value_counts()
rare_categories = val_counts[val_counts < threshold].index
dataset['categories'] = dataset['categories'].apply(
    lambda x: 'other' if x in rare_categories else x
)

In [ ]:
dataset['categories'].value_counts()

,count
categories,
other,14039
astro-ph,6720
hep-ph,2222
quant-ph,1723
hep-th,1525
cond-mat.mtrl-sci,843
gr-qc,804
cond-mat.mes-hall,697
hep-ex,632


In [ ]:
len(dataset['categories'].value_counts())

56

## Create the BertTopic unsupervised Model 1

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L12-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
topics,probs = topic_model.fit_transform(dataset['abstract'])

### Evaluation

In [ ]:
embeddings = topic_model.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.549
CH Index: 45958


### The most frequent topics

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,17299,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...
1,0,535,0_higgs_lhc_quark_quarks,"[higgs, lhc, quark, quarks, bosons, supersymme...",[ We explore the Higgs sector in the supersym...
2,1,499,1_nuclei_neutrons_isotopes_neutron,"[nuclei, neutrons, isotopes, neutron, nucleus,...",[ It is shown that the use of a density depen...
3,2,496,2_curvatures_curvature_riemannian_manifolds,"[curvatures, curvature, riemannian, manifolds,...",[ We study the behaviour of the Ricci Yang-Mi...
4,3,488,3_channels_decoding_channel_decoder,"[channels, decoding, channel, decoder, transmi...",[ We consider a state-dependent full-duplex r...
...,...,...,...,...,...
271,270,16,270_conductance_coulomb_junctions_electron,"[conductance, coulomb, junctions, electron, mo...",[ We outline the qualitatively different phys...
272,271,16,271_dislocation_dislocations_crystallographic_...,"[dislocation, dislocations, crystallographic, ...",[ Due to recent successes of a statistical-ba...
273,272,15,272_stars_star_stellar_supernovae,"[stars, star, stellar, supernovae, luminosity,...",[ [Abridged] We present a comprehensive study...
274,273,15,273_relativistic_relativity_thermodynamics_ent...,"[relativistic, relativity, thermodynamics, ent...",[ We investigate the physical property of the...


### Frequent words from the second most frequent topic

In [ ]:
topic_model.get_topic(0)

[('higgs', np.float32(0.62909603)),
 ('lhc', np.float32(0.5741783)),
 ('quark', np.float32(0.47447482)),
 ('quarks', np.float32(0.46800107)),
 ('bosons', np.float32(0.45707306)),
 ('supersymmetry', np.float32(0.44903964)),
 ('supersymmetric', np.float32(0.41509786)),
 ('lepton', np.float32(0.3498004)),
 ('gauge', np.float32(0.34151113)),
 ('fermion', np.float32(0.33840704))]

### Info about the documents clustered in these topics

In [ ]:
topic_model.get_document_info(dataset['abstract'])

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A fully differential calculation in perturba...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
1,"We describe a new algorithm, the $(k,\ell)$-...",35,35_graphs_graph_subgraphs_subgraph,"[graphs, graph, subgraphs, subgraph, digraph, ...",[ Let C(G) denote the set of lengths of cycle...,graphs - graph - subgraphs - subgraph - digrap...,0.168589,False
2,The evolution of Earth-Moon system is descri...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
3,We show that a determinant of Stirling cycle...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
4,In this paper we show how to compute the $\L...,50,50_hardy_bergman_operators_carleson,"[hardy, bergman, operators, carleson, operator...",[ In this paper we consider the Hardy-Lorentz...,hardy - bergman - operators - carleson - opera...,0.702180,False
...,...,...,...,...,...,...,...,...
39995,Weak scale supersymmetry provides elegant so...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
39996,In this paper we give an alternate combinato...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
39997,"Work of Clifford, Munn and Ponizovski{\u\i} ...",-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False
39998,We present a detailed reconstruction of the ...,-1,-1_galaxies_stars_stellar_star,"[galaxies, stars, stellar, star, emission, gal...",[ (Abridged version) We present the first det...,galaxies - stars - stellar - star - emission -...,0.000000,False


## Data Visualization of the topics

In [ ]:
topic_model.visualize_barchart(top_n_topics=20,n_words=10)

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_term_rank()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topics_per_class = topic_model.topics_per_class(dataset['abstract'], classes=dataset['categories'])
topic_model.visualize_topics_per_class(topics_per_class)

## Create BertTopic supervised model 1

In [ ]:
X,y= dataset.drop(columns=['categories'],axis=1),dataset['categories']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L12-v2")

# Step 2 - Reduce dimensionality
umap_model = BaseDimensionalityReduction()

# Step 3 - Cluster reduced embeddings
hdbscan_model = LogisticRegression()

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [ ]:
X_train = X_train.squeeze().astype(str).tolist()

In [ ]:
y_train = y_train.squeeze().astype(str).tolist()

In [ ]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_train_encoded

array([49, 49,  0, ...,  0,  0,  0])

In [ ]:
topics, probs = topic_model.fit_transform(X_train, y=y_train_encoded)

### Details about the topic frequency

In [ ]:
topic_info_df = topic_model.get_topic_info()
topic_info_df["Class_Label"] = topic_info_df["Topic"].apply(
    lambda t: encoder.inverse_transform([t])[0] if t >= 0 and t < len(encoder.classes_) else "Other"
)
topic_info_df

,Topic,Count,Name,Representation,Representative_Docs,Class_Label
0,0,11172,0_dynamics_quantum_density_numerical,"[dynamics, quantum, density, numerical, theory...",[ The blow-up in finite time for the solution...,astro-ph
1,1,5455,1_galaxies_galactic_stellar_stars,"[galaxies, galactic, stellar, stars, telescope...",[ We present the first high spatial resolutio...,astro-ph hep-ph
2,2,1783,2_quark_lhc_quarks_qcd,"[quark, lhc, quarks, qcd, higgs, bosons, meson...",[ We investigate the associated production of...,cond-mat.dis-nn
3,3,1389,3_entanglement_entangled_quantum_qubits,"[entanglement, entangled, quantum, qubits, qub...",[ A practical scheme for entanglement creatio...,cond-mat.mes-hall
4,4,1216,4_supersymmetry_supersymmetric_supergravity_br...,"[supersymmetry, supersymmetric, supergravity, ...",[ We give a detailed critical discussion of t...,cond-mat.mes-hall cond-mat.mtrl-sci
5,5,665,5_ferromagnetic_magnetization_magnetic_alloys,"[ferromagnetic, magnetization, magnetic, alloy...","[ Theoretical calculations, based on hybrid e...",cond-mat.mes-hall cond-mat.str-el
6,6,640,6_spacetime_spacetimes_gravitational_cosmology,"[spacetime, spacetimes, gravitational, cosmolo...",[ We study the electromagnetic field equation...,cond-mat.mtrl-sci
7,7,556,7_spins_spin_quantum_magnetic,"[spins, spin, quantum, magnetic, electrons, gr...",[ Quantum interference effects in rings provi...,cond-mat.mtrl-sci cond-mat.other
8,8,499,8_mesons_decays_decay_quark,"[mesons, decays, decay, quark, hadronic, proto...",[ We report the observation of charmless hadr...,cond-mat.other
9,9,471,9_neutron_nuclei_nuclear_nucleon,"[neutron, nuclei, nuclear, nucleon, nucleus, p...",[ Thermal properties of asymmetric nuclear ma...,cond-mat.soft


### Frequent words from the second most frequent topic

In [ ]:
topic_model.get_topic(0)

[('dynamics', np.float32(0.44298545)),
 ('quantum', np.float32(0.3275764)),
 ('density', np.float32(0.30828282)),
 ('numerical', np.float32(0.24250662)),
 ('theory', np.float32(0.23960298)),
 ('phase', np.float32(0.21765576)),
 ('spin', np.float32(0.21119249)),
 ('energy', np.float32(0.20504196)),
 ('lattice', np.float32(0.20408583)),
 ('temperature', np.float32(0.20305517))]

## Data visualization of the topics

In [ ]:
topic_model.visualize_barchart(top_n_topics=20,n_words=10)

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
X_test = X_test.squeeze().astype(str).tolist()
predicted_topics, _ = topic_model.transform(X_test)

In [ ]:
predicted_labels = encoder.inverse_transform(predicted_topics)

In [ ]:
y_test = y_test.squeeze().astype(str).tolist()
print(classification_report(y_test,predicted_labels,zero_division=0))

                                     precision    recall  f1-score   support

                           astro-ph       0.02      0.05      0.03      1265
                    astro-ph hep-ph       0.01      0.60      0.02        20
                    cond-mat.dis-nn       0.00      0.00      0.00        15
                  cond-mat.mes-hall       0.02      0.04      0.02       141
cond-mat.mes-hall cond-mat.mtrl-sci       0.00      0.00      0.00        23
  cond-mat.mes-hall cond-mat.str-el       0.01      0.06      0.02        32
                  cond-mat.mtrl-sci       0.00      0.00      0.00       178
   cond-mat.mtrl-sci cond-mat.other       0.01      0.09      0.02        22
                     cond-mat.other       0.00      0.00      0.00        96
                      cond-mat.soft       0.00      0.00      0.00        59
   cond-mat.soft cond-mat.stat-mech       0.00      0.00      0.00        35
                 cond-mat.stat-mech       0.50      0.22      0.30       11

## Create BertTopic Model unsupervised 2

In [ ]:
embedding_model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_components=5, metric='cosine', random_state=42)

# Step 3 - Cluster reduced embeddings
hdbscan_model = KMeans()

# Step 4 - Tokenize topics
vectorizer_model = TfidfVectorizer(stop_words="english")


# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = MaximalMarginalRelevance(diversity=0.3)

# All steps together
topic_model_2 = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
topics, _ = topic_model_2.fit_transform(dataset['abstract'])

### Evaluation

In [ ]:
embeddings = topic_model_2.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.442
CH Index: 44833


### The most frequent topics

In [ ]:
topic_model_2.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,7716,0_algebras_finite_groups_spaces,"[algebras, finite, groups, spaces, cohomology,...",[ We consider algebras in a modular tensor ca...
1,1,6481,1_galaxies_stars_galaxy_stellar,"[galaxies, stars, galaxy, stellar, ngc, galact...",[ A 10-arcmin field around the HDF(N) contain...
2,2,6128,2_spin_magnetic_electron_quantum,"[spin, magnetic, electron, quantum, graphene, ...",[ The insulating magnetic phase in graphene z...
3,3,4978,3_quark_neutrino_qcd_higgs,"[quark, neutrino, qcd, higgs, decays, lhc, gam...",[ B decays are a subject of active research s...
4,4,4699,4_quantum_entanglement_dynamics_phase,"[quantum, entanglement, dynamics, phase, equat...",[ In the present paper we study the entanglem...
5,5,4436,5_paper_network_model_data,"[paper, network, model, data, algorithm, probl...",[ Many important real-world networks manifest...
6,6,3167,6_theory_gauge_quantum_equations,"[theory, gauge, quantum, equations, fields, sy...","[ The $sp(8, R)$ invariant formulation of fre..."
7,7,2395,7_cosmological_matter_gravity_gravitational,"[cosmological, matter, gravity, gravitational,...",[ We study non-linear structure formation in ...


### Frequent words from the second most frequent topic

In [ ]:
topic_model_2.get_topic(0)

[('algebras', np.float64(0.01755437002489335)),
 ('finite', np.float64(0.017468134158072624)),
 ('groups', np.float64(0.015786288861467396)),
 ('spaces', np.float64(0.015054440666585884)),
 ('cohomology', np.float64(0.014489989475501166)),
 ('theorem', np.float64(0.01415379475693976)),
 ('lie', np.float64(0.012777469126203754)),
 ('theory', np.float64(0.012580671724332571)),
 ('complex', np.float64(0.011908767830186325)),
 ('manifold', np.float64(0.010943386121941137))]

### Info about the documents clustered in these topics

In [ ]:
topic_model_2.get_document_info(dataset['abstract'])

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,A fully differential calculation in perturba...,3,3_quark_neutrino_qcd_higgs,"[quark, neutrino, qcd, higgs, decays, lhc, gam...",[ B decays are a subject of active research s...,quark - neutrino - qcd - higgs - decays - lhc ...,False
1,"We describe a new algorithm, the $(k,\ell)$-...",0,0_algebras_finite_groups_spaces,"[algebras, finite, groups, spaces, cohomology,...",[ We consider algebras in a modular tensor ca...,algebras - finite - groups - spaces - cohomolo...,False
2,The evolution of Earth-Moon system is descri...,1,1_galaxies_stars_galaxy_stellar,"[galaxies, stars, galaxy, stellar, ngc, galact...",[ A 10-arcmin field around the HDF(N) contain...,galaxies - stars - galaxy - stellar - ngc - ga...,False
3,We show that a determinant of Stirling cycle...,5,5_paper_network_model_data,"[paper, network, model, data, algorithm, probl...",[ Many important real-world networks manifest...,paper - network - model - data - algorithm - p...,False
4,In this paper we show how to compute the $\L...,0,0_algebras_finite_groups_spaces,"[algebras, finite, groups, spaces, cohomology,...",[ We consider algebras in a modular tensor ca...,algebras - finite - groups - spaces - cohomolo...,False
...,...,...,...,...,...,...,...
39995,Weak scale supersymmetry provides elegant so...,3,3_quark_neutrino_qcd_higgs,"[quark, neutrino, qcd, higgs, decays, lhc, gam...",[ B decays are a subject of active research s...,quark - neutrino - qcd - higgs - decays - lhc ...,False
39996,In this paper we give an alternate combinato...,0,0_algebras_finite_groups_spaces,"[algebras, finite, groups, spaces, cohomology,...",[ We consider algebras in a modular tensor ca...,algebras - finite - groups - spaces - cohomolo...,False
39997,"Work of Clifford, Munn and Ponizovski{\u\i} ...",0,0_algebras_finite_groups_spaces,"[algebras, finite, groups, spaces, cohomology,...",[ We consider algebras in a modular tensor ca...,algebras - finite - groups - spaces - cohomolo...,False
39998,We present a detailed reconstruction of the ...,1,1_galaxies_stars_galaxy_stellar,"[galaxies, stars, galaxy, stellar, ngc, galact...",[ A 10-arcmin field around the HDF(N) contain...,galaxies - stars - galaxy - stellar - ngc - ga...,False


## Visualization of the topics

In [ ]:
topic_model_2.visualize_barchart(top_n_topics=20,n_words=8)

In [ ]:
topic_model_2.visualize_heatmap()

In [ ]:
topic_model_2.visualize_topics()

In [ ]:
topic_model_2.visualize_term_rank()

In [ ]:
topic_model_2.visualize_hierarchy()

In [ ]:
topics_per_class = topic_model_2.topics_per_class(dataset['abstract'], classes=dataset['categories'])
topic_model_2.visualize_topics_per_class(topics_per_class)